# Fine-tune a RAVE model on Google Colab

This notebook continues the training of an existing [RAVE](https://github.com/acids-ircam/RAVE) model on new
sounds, keeping everything on your Google Drive so that it **survives Colab disconnections**, then exports the
result for [`nn~`](https://github.com/acids-ircam/nn_tilde) and the RAVE VST.

Typical uses:

- **`encoder` mode** (default): the model keeps its sound (decoder fixed) but learns to *understand* a new kind of
  input. For instance, fine-tune a flute model on voice recordings to improve voice-to-flute timbre transfer.
- **`decoder` mode**: keep the latent space, adapt the sound to the new data.
- **`full` / `alternating` modes**: adapt the whole model to a new dataset (faster than training from scratch).

The source model can be a training run folder or a `.ckpt` on your Drive, or an exported `.ts` (it is then
*uncompiled* first; its discriminator is lost, so `decoder` mode is not available, and `full` / `alternating`
re-initialize it).

Everything is written in `MyDrive/RAVE`:

```
RAVE/
├── audio/<experiment>/          audio files of the new dataset
├── datasets/                    preprocessed datasets (computed once) and their train / val splits
├── finetunes/<experiment>/      fine-tuning run: checkpoints, tensorboard logs, config
├── finetunes/<experiment>.json  settings of the experiment
└── exports/                     exported models (.ts)
```

See the [fine-tuning tutorial](https://acids-rave.readthedocs.io/en/latest/tutorials/07_finetune/) for details.

## If Colab disconnects

Free Colab sessions stop regularly (idle timeout, usage limits). **Nothing is lost**: runs, checkpoints, the
preprocessed dataset and its train / validation split are stored on your Drive. To continue:

1. Reconnect (or wait until a GPU is available again);
2. **Runtime > Run all**, with the same settings.

The notebook finds the existing run and resumes from its most advanced readable checkpoint. A checkpoint is
written at each validation (`VAL_EVERY` steps): you lose at most the steps done since the last one.
Changing a fine-tuning setting (mode, learning rates, source model...) is refused: use another experiment name to start a new
model.

Keep the browser tab open while training: closing it, or leaving it idle for a long time, ends the session sooner.

In [ ]:
#@title Settings
#@markdown **Experiment**: changing the name starts a new fine-tuning.
EXPERIMENT = "my_finetune" #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/RAVE" #@param {type:"string"}
#@markdown Source model on Drive: run folder, `.ckpt` or exported `.ts`.
SOURCE_MODEL = "/content/drive/MyDrive/RAVE/runs/my_model_0123456789" #@param {type:"string"}
#@markdown Audio folder of the new dataset (empty: `DRIVE_ROOT/audio/EXPERIMENT`).
AUDIO_FOLDER = "" #@param {type:"string"}

#@markdown **Fine-tuning** (cannot be changed once started).
MODE = "encoder" #@param ["encoder", "decoder", "full", "alternating"]
LR_ENCODER = 1e-5 #@param {type:"number"}
LR_DECODER = 1e-4 #@param {type:"number"}
LR_DISCRIMINATOR = 1e-4 #@param {type:"number"}
#@markdown Number of fine-tuning steps during which the encoder stays fixed.
ENCODER_START_STEP = 0 #@param {type:"integer"}
#@markdown KL weight of the encoder (variational models). Empty: source value / 4.
BETA = "" #@param {type:"string"}
#@markdown Restrict trained / frozen weights: `encoder`, `decoder`, `discriminator` or regular expressions, comma separated. Empty: no restriction.
TRAIN_WEIGHTS = "" #@param {type:"string"}
FREEZE_WEIGHTS = "" #@param {type:"string"}
#@markdown Re-initialize the discriminator (and train it).
RESET_DISCRIMINATOR = False #@param {type:"boolean"}
#@markdown Recompute the latent PCA during fine-tuning. Untick to keep the latent axes of the source model.
UPDATE_PCA = True #@param {type:"boolean"}
#@markdown Data augmentations (comma separated). Keep `mute`.
AUGMENTATIONS = "mute,compress,gain" #@param {type:"string"}

#@markdown **Training** (can be changed when resuming).
BATCH = 8 #@param {type:"integer"}
#@markdown Total number of fine-tuning steps.
MAX_STEPS = 200000 #@param {type:"integer"}
#@markdown A checkpoint is saved at each validation: lower values lose less progress on disconnection.
VAL_EVERY = 5000 #@param {type:"integer"}

import os, sys
from pathlib import Path
if "/content" not in sys.path: sys.path.append("/content")
assert EXPERIMENT and all(c.isalnum() or c in "-_" for c in EXPERIMENT), "use only letters, digits, - and _ in EXPERIMENT"
AUDIO_FOLDER = AUDIO_FOLDER or f"{DRIVE_ROOT}/audio/{EXPERIMENT}"
FT_DIR = Path(DRIVE_ROOT) / "finetunes"
RUN_DIR = FT_DIR / EXPERIMENT
EXPORT_DIR = Path(DRIVE_ROOT) / "exports"
AUGMENT_LIST = [a.strip() for a in AUGMENTATIONS.split(",") if a.strip()]
split = lambda s: [w.strip() for w in s.split(",") if w.strip()]

## Setup
Run these cells on every new session.

In [ ]:
#@title Check the GPU and mount Google Drive
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError("No GPU: select Runtime > Change runtime type > T4 GPU, then run all cells again.")
print("GPU:", gpu.stdout.strip())

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
#@title Install RAVE
#@markdown RAVE 2.4 is installed from GitHub. Re-run on every new session (Colab machines start empty).
RAVE_SOURCE = "git+https://github.com/acids-ircam/RAVE@v2.4" #@param {type:"string"}
import importlib.util, subprocess, sys
subprocess.run(["apt-get", "-qq", "install", "-y", "sox", "libsox-dev"], check=False,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", RAVE_SOURCE], check=True)

# torchcodec (audio decoding) must match the installed torch version: fall back to older releases if needed
def torchcodec_ok():
    return subprocess.run([sys.executable, "-c", "from torchcodec.decoders import AudioDecoder"],
                          capture_output=True).returncode == 0
if not torchcodec_ok():
    for version in ["0.8.*", "0.7.*", "0.6.*", "0.5.*", "0.4.*", "0.3.*", "0.2.*"]:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", f"torchcodec=={version}"],
                       check=False, capture_output=True)
        if torchcodec_ok():
            print("using torchcodec", version)
            break
    else:
        print("WARNING: torchcodec does not work with this torch version; audio decoding may fail.")

subprocess.run(["rave", "--help"], capture_output=True, check=True)
print(subprocess.run([sys.executable, "-c", "import torch, rave; print('torch', torch.__version__, '| rave ok')"],
                     capture_output=True, text=True).stdout)

Helper functions (Drive persistence, dataset cache, checkpoint search). Nothing to change here.

In [ ]:
%%writefile /content/rave_colab.py
"""Helpers of the RAVE Colab notebooks: Drive persistence, local dataset cache and safe resuming.

Everything that must survive a disconnection (runs, checkpoints, preprocessed datasets, partitions, settings)
lives on Google Drive. The dataset is copied to the local disk of the Colab machine for fast reading.
"""
import json, os, re, shlex, shutil, signal, subprocess, sys, tarfile, threading, time
from pathlib import Path

LOCAL_ROOT = Path("/content/rave_local")
PROGRESS_EVERY = 60  # seconds between two printed progress bar updates


def run(cmd, progress_every=PROGRESS_EVERY):
    """Runs a command, streams its output (progress bars are throttled to keep the notebook light) and returns its
    exit code. Interrupting the cell stops the command cleanly."""
    cmd = [str(c) for c in cmd]
    print("$ " + " ".join(shlex.quote(c) for c in cmd), flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
    buf, last_progress = "", 0.
    try:
        while True:
            chunk = os.read(p.stdout.fileno(), 65536)
            if not chunk:
                break
            buf += chunk.decode(errors="replace")
            while True:
                idx = [i for i in (buf.find("\n"), buf.find("\r")) if i >= 0]
                if not idx:
                    break
                i = min(idx)
                line, sep, buf = buf[:i], buf[i], buf[i + 1:]
                if sep == "\n":
                    print(line, flush=True)
                elif line.strip() and time.time() - last_progress > progress_every:
                    print(line, flush=True)
                    last_progress = time.time()
        if buf.strip():
            print(buf, flush=True)
    except KeyboardInterrupt:
        print("\nInterrupted: stopping the process (this can take a minute)...", flush=True)
        p.send_signal(signal.SIGINT)
        try:
            p.wait(timeout=120)
        except subprocess.TimeoutExpired:
            p.kill()
        raise
    return p.wait()


def check(cmd, **kwargs):
    code = run(cmd, **kwargs)
    if code != 0:
        raise RuntimeError(f"command failed (exit code {code}), see the output above")


# ----------------------------------------------------------------- settings --

def lock_settings(path, settings):
    """Saves the settings of an experiment on Drive the first time, and checks on later sessions that they did not
    change (a resumed training must use the same model / data settings)."""
    path = Path(path)
    if path.exists():
        saved = json.loads(path.read_text())
        diff = {k: (saved.get(k), v) for k, v in settings.items() if saved.get(k) != v}
        if diff:
            lines = "\n".join(f"  {k}: saved {a!r}, now {b!r}" for k, (a, b) in diff.items())
            raise ValueError(f"These settings changed since this experiment was started ({path}):\n{lines}\n"
                             "Restore them to resume the training, or choose another experiment name to start a new one.")
        print(f"Settings match the saved settings of this experiment ({path.name}).")
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp = path.with_suffix(".tmp")
        tmp.write_text(json.dumps(settings, indent=2))
        tmp.rename(path)
        print(f"New experiment: settings saved to {path}")


# ------------------------------------------------------------------ dataset --

def _copy_audio(src, dst):
    exts = {".wav", ".aif", ".aiff", ".flac", ".mp3", ".ogg", ".opus", ".m4a", ".aac"}
    n = 0
    for f in Path(src).rglob("*"):
        if f.is_file() and f.suffix.lower() in exts:
            target = Path(dst) / f.relative_to(src)
            if not target.exists() or target.stat().st_size != f.stat().st_size:
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, target)
            n += 1
    return n


def _link_partitions(local_db, drive_partitions):
    """Train / validation splits are written in <db>/partitions: keep them on Drive, so that a resumed training
    reuses the same split (the local dataset copy disappears with the Colab machine)."""
    drive_partitions.mkdir(parents=True, exist_ok=True)
    local_parts = local_db / "partitions"
    if local_parts.is_symlink():
        return
    if local_parts.exists():
        for f in local_parts.iterdir():
            if not (drive_partitions / f.name).exists():
                shutil.copy2(f, drive_partitions / f.name)
        shutil.rmtree(local_parts)
    local_parts.symlink_to(drive_partitions, target_is_directory=True)


def prepare_dataset(audio_dir, drive_root, dataset_name, channels, sampling_rate, num_signal=131072):
    """Returns the path of a local, preprocessed copy of the dataset. The preprocessed dataset is stored on Drive
    as a single archive, so that it is only computed once: later sessions just copy and extract it."""
    drive_ds = Path(drive_root) / "datasets"
    archive = drive_ds / f"{dataset_name}.tar"
    local_db = LOCAL_ROOT / "datasets" / dataset_name
    ready = local_db / ".ready"
    if ready.exists():
        print(f"Dataset already available on this machine: {local_db}")
    elif archive.exists():
        print(f"Copying preprocessed dataset from Drive ({archive.stat().st_size / 1e9:.2f} GB)...", flush=True)
        shutil.rmtree(local_db, ignore_errors=True)
        local_db.mkdir(parents=True)
        tmp = LOCAL_ROOT / f"{dataset_name}.tar"
        shutil.copyfile(archive, tmp)
        with tarfile.open(tmp) as tar:
            tar.extractall(local_db, filter="data")
        tmp.unlink()
        ready.touch()
    else:
        if not Path(audio_dir).is_dir():
            raise FileNotFoundError(f"audio folder {audio_dir} not found: upload your audio files there.")
        local_audio = LOCAL_ROOT / "audio" / dataset_name
        print(f"Copying audio files from {audio_dir} to the local disk...", flush=True)
        n = _copy_audio(audio_dir, local_audio)
        if n == 0:
            raise FileNotFoundError(f"no audio file found in {audio_dir}")
        print(f"{n} audio files. Preprocessing...", flush=True)
        shutil.rmtree(local_db, ignore_errors=True)
        check(["rave", "preprocess", "--input_path", local_audio, "--output_path", local_db,
               "--channels", channels, "--sampling_rate", sampling_rate, "--num_signal", num_signal])
        print("Saving the preprocessed dataset to Drive (only done once)...", flush=True)
        tmp = LOCAL_ROOT / f"{dataset_name}.tar"
        with tarfile.open(tmp, "w") as tar:
            for f in local_db.iterdir():
                if f.name != "partitions":
                    tar.add(f, arcname=f.name)
        drive_ds.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(tmp, archive.with_suffix(".tar.tmp"))
        archive.with_suffix(".tar.tmp").rename(archive)  # never leave a half-written archive on Drive
        tmp.unlink()
        ready.touch()
    _link_partitions(local_db, drive_ds / f"{dataset_name}_partitions")
    return local_db


# -------------------------------------------------------------- checkpoints --

_SCAN = r"""
import json, os, sys, torch
from pathlib import Path
out = []
for root in sys.argv[1:]:
    for p in Path(root).rglob("*.ckpt"):
        try:
            c = torch.load(p, map_location="cpu", mmap=True, weights_only=False)
            ft, gs = c.get("finetune"), int(c.get("global_step", 0))
            step = int(ft["steps"]) if isinstance(ft, dict) and "steps" in ft else gs
            out.append({"path": str(p), "step": step, "mtime": os.path.getmtime(p)})
        except Exception as e:
            out.append({"path": str(p), "error": f"{type(e).__name__}: {e}"})
print("@@SCAN@@" + json.dumps(out))
"""


def scan_checkpoints(*roots):
    """Reads the training step of every checkpoint under roots. Unreadable checkpoints (e.g. cut by a
    disconnection during a write) are reported and ignored."""
    roots = [str(r) for r in roots if Path(r).exists()]
    if not roots:
        return []
    out = subprocess.run([sys.executable, "-c", _SCAN, *roots], capture_output=True, text=True)
    lines = [l for l in out.stdout.splitlines() if l.startswith("@@SCAN@@")]
    if not lines:
        raise RuntimeError("could not scan checkpoints:\n" + out.stderr[-3000:])
    found = json.loads(lines[-1][len("@@SCAN@@"):])
    for c in found:
        if "error" in c:
            print(f"  ignoring unreadable checkpoint {c['path']} ({c['error'][:120]})")
    return [c for c in found if "error" not in c]


def latest_checkpoint(*roots):
    """Most advanced readable checkpoint (highest step, then most recent). Each resumed session writes its own
    version_N folder with its own last.ckpt, so file names alone do not tell which checkpoint is the latest."""
    ckpts = scan_checkpoints(*roots)
    if not ckpts:
        return None
    return max(ckpts, key=lambda c: (c["step"], c["mtime"]))


def find_training_runs(runs_dir, name):
    """Run folders of `rave train --name name` (<name>_<10 hex characters>)."""
    pattern = re.compile(re.escape(name) + r"_[0-9a-f]{10}")
    runs_dir = Path(runs_dir)
    if not runs_dir.exists():
        return []
    return [p for p in runs_dir.iterdir() if p.is_dir() and pattern.fullmatch(p.name)]


# ------------------------------------------------------------- drive trash --

def purge_trashed_checkpoints(dry_run=True):
    """Checkpoints replaced during training are moved to the Drive trash, which counts in your storage quota.
    Permanently deletes trashed files named *.ckpt whose folder is named "checkpoints" (RAVE checkpoints).
    Nothing else in your trash is touched. Requires the Drive authorization of authorize_drive_api()."""
    from googleapiclient.discovery import build
    service = build("drive", "v3", cache_discovery=False)
    folder_names, freed, n, token = {}, 0, 0, None
    while True:
        res = service.files().list(q="trashed = true and name contains '.ckpt'", spaces="drive", pageToken=token,
                                   fields="nextPageToken, files(id, name, size, parents)", pageSize=200).execute()
        for f in res.get("files", []):
            if not f["name"].endswith(".ckpt"):
                continue
            parents = f.get("parents", [])
            names = []
            for pid in parents:
                if pid not in folder_names:
                    try:
                        folder_names[pid] = service.files().get(fileId=pid, fields="name").execute()["name"]
                    except Exception:
                        folder_names[pid] = None
                names.append(folder_names[pid])
            if "checkpoints" not in names:
                continue
            size = int(f.get("size", 0))
            if not dry_run:
                service.files().delete(fileId=f["id"]).execute()
            freed += size; n += 1
        token = res.get("nextPageToken")
        if token is None:
            break
    verb = "would free" if dry_run else "freed"
    print(f"{n} trashed RAVE checkpoints: {verb} {freed / 1e9:.2f} GB")
    return n, freed


def authorize_drive_api():
    from google.colab import auth
    auth.authenticate_user()


def start_trash_purger(every_minutes=30):
    """Purges trashed RAVE checkpoints in the background while training."""
    def loop():
        while True:
            time.sleep(every_minutes * 60)
            try:
                purge_trashed_checkpoints(dry_run=False)
            except Exception as e:
                print(f"[trash purge failed: {e}]")
    threading.Thread(target=loop, daemon=True).start()
    print(f"Trashed RAVE checkpoints will be deleted every {every_minutes} minutes.")

### (Optional) Keep your Drive from filling up

At each validation, RAVE replaces two or three checkpoints of a few hundred MB each. On a mounted Drive, replaced
files go to the **Drive trash, which still counts in your storage quota**: within a few hours this can fill a
free Drive (15 GB), and checkpoints can then no longer be saved.

With `AUTO_PURGE` (default), trashed RAVE checkpoints (files `*.ckpt` in a folder named `checkpoints`) are
permanently deleted now and every 30 minutes during training. Nothing else in your trash is touched. This asks
for an additional Drive authorization. Untick it if you prefer to empty the trash yourself on drive.google.com.

In [ ]:
#@title Drive trash
AUTO_PURGE = True #@param {type:"boolean"}
import rave_colab
if AUTO_PURGE:
    rave_colab.authorize_drive_api()
    rave_colab.purge_trashed_checkpoints(dry_run=False)
    rave_colab.start_trash_purger(every_minutes=30)

## Source model and dataset

In [ ]:
#@title Load the source model
import json, subprocess, sys
import rave_colab
rave_colab.lock_settings(FT_DIR / f"{EXPERIMENT}.json", {
    "source_model": SOURCE_MODEL, "mode": MODE, "lr_encoder": LR_ENCODER, "lr_decoder": LR_DECODER,
    "lr_discriminator": LR_DISCRIMINATOR, "encoder_start_step": ENCODER_START_STEP, "beta": BETA,
    "train": TRAIN_WEIGHTS, "freeze": FREEZE_WEIGHTS, "reset_discriminator": RESET_DISCRIMINATOR,
    "update_pca": UPDATE_PCA, "augmentations": AUGMENT_LIST, "audio_folder": AUDIO_FOLDER,
})
if not Path(SOURCE_MODEL).exists():
    raise FileNotFoundError(f"source model {SOURCE_MODEL} not found")

if SOURCE_MODEL.endswith(".ts"):
    if MODE == "decoder":
        raise ValueError("decoder mode needs the discriminator of the source model, which exported .ts files do "
                         "not contain: use the training run, or another mode.")
    if MODE in ("full", "alternating") and not RESET_DISCRIMINATOR:
        raise ValueError(f"{MODE} mode with a .ts source: tick RESET_DISCRIMINATOR (the .ts has no discriminator).")
    SOURCE_RUN = str(FT_DIR / f"{EXPERIMENT}_source")
    if not (Path(SOURCE_RUN) / "last.ckpt").exists():
        print("Uncompiling the .ts into a trainable checkpoint (done once)...")
        rave_colab.check(["rave", "uncompile", "--model", SOURCE_MODEL, "--out", SOURCE_RUN])
    else:
        print("Using the uncompiled source model", SOURCE_RUN)
    SOURCE_RUN = str(Path(SOURCE_RUN) / "last.ckpt")
elif Path(SOURCE_MODEL).is_dir():
    # use the most advanced checkpoint of the run explicitly (a run folder holds one folder per training session)
    source_latest = rave_colab.latest_checkpoint(SOURCE_MODEL)
    if source_latest is None:
        raise FileNotFoundError(f"no readable checkpoint in {SOURCE_MODEL}")
    SOURCE_RUN = source_latest["path"]
    print(f"source checkpoint: step {source_latest['step']}")
else:
    SOURCE_RUN = SOURCE_MODEL

# sampling rate and channels of the source model (the new dataset is preprocessed accordingly)
probe = subprocess.run([sys.executable, "-c",
    "import json, sys, rave; m, p = rave.load_rave_checkpoint(sys.argv[1]); "
    "print('@@' + json.dumps({'sr': int(m.sr), 'channels': int(m.n_channels), 'ckpt': str(p)}))", SOURCE_RUN],
    capture_output=True, text=True)
info = [l for l in probe.stdout.splitlines() if l.startswith("@@")]
if not info:
    raise RuntimeError("could not load the source model:\n" + probe.stderr[-3000:])
SOURCE_INFO = json.loads(info[-1][2:])
print(f"source model: {SOURCE_INFO['ckpt']} ({SOURCE_INFO['sr']} Hz, {SOURCE_INFO['channels']} channel(s))")

In [ ]:
#@title Prepare the dataset
#@markdown The first time, your audio is preprocessed and saved to `DRIVE_ROOT/datasets`. Later sessions only copy it.
sr, ch = SOURCE_INFO["sr"], SOURCE_INFO["channels"]
DB_PATH = rave_colab.prepare_dataset(AUDIO_FOLDER, DRIVE_ROOT, f"{EXPERIMENT}_{sr}hz_{ch}ch",
                                     channels=ch, sampling_rate=sr)
print("dataset:", DB_PATH)

## Fine-tuning

In [ ]:
#@title Monitor with TensorBoard
%load_ext tensorboard
%tensorboard --logdir "$FT_DIR"

In [ ]:
#@title Fine-tune (or resume)
#@markdown Runs until `MAX_STEPS`, or until you stop the cell / Colab disconnects. Run it again to resume.
latest = rave_colab.latest_checkpoint(RUN_DIR)

cmd = ["rave", "finetune", "--name", EXPERIMENT, "--run", SOURCE_RUN, "--db_path", DB_PATH, "--out_path", FT_DIR,
       "--mode", MODE, "--lr_encoder", LR_ENCODER, "--lr_decoder", LR_DECODER,
       "--lr_discriminator", LR_DISCRIMINATOR, "--encoder_start_step", ENCODER_START_STEP,
       "--batch", BATCH, "--max_steps", MAX_STEPS, "--val_every", VAL_EVERY, "--workers", 2, "--device", "cuda:0",
       "--update_pca" if UPDATE_PCA else "--noupdate_pca"]
if BETA.strip(): cmd += ["--beta", float(BETA)]
for w in split(TRAIN_WEIGHTS): cmd += ["--train", w]
for w in split(FREEZE_WEIGHTS): cmd += ["--freeze", w]
if RESET_DISCRIMINATOR: cmd += ["--reset_discriminator"]
for a in AUGMENT_LIST: cmd += ["--augment", a]

if latest is not None:
    print(f"Resuming from fine-tuning step {latest['step']}: {latest['path']}")
    if latest["step"] >= MAX_STEPS:
        print(f"MAX_STEPS ({MAX_STEPS}) already reached: increase it to continue fine-tuning.")
        cmd = None
    else:
        cmd += ["--resume", latest["path"]]
else:
    print("Starting a new fine-tuning.")

if cmd is not None:
    code = rave_colab.run(cmd)
    if code != 0:
        print(f"\nFine-tuning stopped with an error (exit code {code}). If the error is 'CUDA out of memory', "
              "lower BATCH. Otherwise, run this cell again to resume from the last checkpoint.")

## Export

Run this when you want to try the model (you can stop the training cell first, and resume it afterwards). The
latest checkpoint is exported in streaming mode, for `nn~` and the RAVE VST, with its latent PCA recomputed on
your dataset. Exported models are saved to `DRIVE_ROOT/exports`.

In [ ]:
#@title Export the latest checkpoint
#@markdown Fraction of the latent variance kept (the number of latent dimensions controllable in `nn~`).
FIDELITY = 0.95 #@param {type:"number"}
#@markdown Recompute the latent PCA on the new dataset. Untick to keep the PCA of the checkpoint (e.g. to keep the latent axes of the source model with `UPDATE_PCA` off).
RECOMPUTE_PCA = True #@param {type:"boolean"}
#@markdown Number of dataset chunks used to compute the PCA (0: all).
MAX_CHUNKS = 2000 #@param {type:"integer"}
DOWNLOAD = True #@param {type:"boolean"}

latest = rave_colab.latest_checkpoint(RUN_DIR)
assert latest is not None, "no checkpoint to export yet"
print(f"Exporting fine-tuning step {latest['step']}: {latest['path']}")
cmd = ["rave", "export", "--run", latest["path"], "--streaming", "--name", f"{EXPERIMENT}_step{latest['step']}",
       "--out_path", EXPORT_DIR, "--fidelity", FIDELITY, "--device", "cuda:0"]
if RECOMPUTE_PCA:
    cmd += ["--db_path", DB_PATH]
    if MAX_CHUNKS > 0: cmd += ["--max_chunks", MAX_CHUNKS]
rave_colab.check(cmd)

exported = max(EXPORT_DIR.glob(f"{EXPERIMENT}_step{latest['step']}*.ts"), key=os.path.getmtime)
print("exported to", exported)
if DOWNLOAD:
    from google.colab import files
    files.download(str(exported))